# Pneumonia Detection v5 — Kermany + RSNA Cross-Dataset Study
### MC Dropout + Calibration + Uncertainty as Distribution Shift Signal

**Research framing (stronger than single-dataset):**
> *"A DenseNet-121 model trained on curated single-site data (Kermany) is evaluated on
> real-world multi-site data (RSNA). MC Dropout uncertainty is used to detect distribution
> shift — cases where the model encounters imaging conditions unseen during training.
> We show that uncertainty-based deferral meaningfully improves clinical reliability
> when the model is deployed out-of-distribution."*

**Why this combination works for uncertainty:**
- Kermany = clean, single-hospital, curated → model trains confidently
- RSNA = 16 institutions, real ED workflow, noisy labels → model encounters genuine ambiguity
- Uncertainty should spike on RSNA images → triage simulation becomes meaningful
- Cross-dataset evaluation is a standard published methodology (domain generalisation)

**Dataset sizes:**
- Kermany: 5,216 images (already have)
- RSNA: ~26,000 DICOM images, ~6GB (downloaded via kagglehub)
---

## Cell 1 — Setup

In [ ]:
import torch, os, warnings
warnings.filterwarnings("ignore")
print("GPU:", torch.cuda.is_available())
!pip install -q pydicom opencv-python-headless scikit-learn matplotlib pandas kagglehub

## Cell 2 — Download datasets
Kermany is already stored locally. RSNA is downloaded via kagglehub.
RSNA images are in DICOM format — we convert them to PNG on the fly.

In [ ]:
import kagglehub

# ── Kermany (already cached) ──────────────────────────────────────────────
KERMANY_BASE = "/kaggle/input/chest-xray-pneumonia/chest_xray"
print("Kermany exists:", os.path.exists(KERMANY_BASE))

# ── RSNA (download) ───────────────────────────────────────────────────────
rsna_path = kagglehub.competition_download('rsna-pneumonia-detection-challenge')
print("Path to competition files:", rsna_path)

RSNA_TRAIN_DIR = os.path.join(rsna_path, "stage_2_train_images")
RSNA_CSV       = os.path.join(rsna_path, "stage_2_detailed_class_info.csv")
RSNA_LABELS    = os.path.join(rsna_path, "stage_2_train_labels.csv")

print("RSNA train images:", os.path.exists(RSNA_TRAIN_DIR))
print("RSNA class CSV   :", os.path.exists(RSNA_CSV))
print("RSNA label CSV   :", os.path.exists(RSNA_LABELS))
print("RSNA image count :", len(os.listdir(RSNA_TRAIN_DIR)) if os.path.exists(RSNA_TRAIN_DIR) else "NOT FOUND")

## Cell 3 — Load Kermany (training data)
Kermany provides the training distribution. We use all its train split for training
and its test split as in-distribution (ID) test set for baseline comparison.

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from sklearn.model_selection import train_test_split

# ── Transforms ────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ── Kermany splits ────────────────────────────────────────────────────────
# raw_tf base — TransformSubset applies train_tf or val_tf exactly once
raw_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset; self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, i):
        img, label = self.subset[i]
        return self.transform(img), label

train_full   = ImageFolder(os.path.join(KERMANY_BASE, "train"), transform=raw_tf)
kermany_test = ImageFolder(os.path.join(KERMANY_BASE, "test"),  transform=raw_tf)

print("Class map:", train_full.class_to_idx)  # NORMAL=0, PNEUMONIA=1

# 85/15 stratified split for train/val
all_targets = train_full.targets
tr_idx, val_idx = train_test_split(
    range(len(train_full)), test_size=0.15,
    stratify=all_targets, random_state=42)

train_ds      = TransformSubset(Subset(train_full, tr_idx),  train_tf)
val_ds        = TransformSubset(Subset(train_full, val_idx), val_tf)
kermany_test_ds = TransformSubset(
    Subset(kermany_test, range(len(kermany_test))), val_tf)

train_loader       = DataLoader(train_ds,        batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader         = DataLoader(val_ds,          batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
kermany_test_loader= DataLoader(kermany_test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Kermany test: {len(kermany_test_ds)}")

## Cell 4 — Build RSNA test dataset (out-of-distribution)
RSNA has three classes: Normal, Lung Opacity (pneumonia), No Lung Opacity/Not Normal.
We map: Lung Opacity → 1 (pneumonia), Normal → 0.
We drop "No Lung Opacity/Not Normal" — these are ambiguous non-pneumonia cases
that would add noise to binary evaluation.

Images are DICOM format. We convert to RGB numpy arrays on the fly using pydicom.

In [ ]:
import pandas as pd
import pydicom
from PIL import Image
import numpy as np

# ── Load RSNA labels ──────────────────────────────────────────────────────
class_df  = pd.read_csv(RSNA_CSV)
labels_df = pd.read_csv(RSNA_LABELS)

print("RSNA class distribution:")
print(class_df["class"].value_counts())
print()

# Merge class info with labels
rsna_df = class_df.merge(
    labels_df[["patientId","Target"]].drop_duplicates("patientId"),
    on="patientId", how="left")

# Keep only Normal and Lung Opacity (binary task matching Kermany)
rsna_binary = rsna_df[rsna_df["class"].isin(["Normal","Lung Opacity"])].copy()
rsna_binary["label"] = (rsna_binary["class"] == "Lung Opacity").astype(int)
rsna_binary = rsna_binary.drop_duplicates("patientId").reset_index(drop=True)

print(f"RSNA binary — Normal: {(rsna_binary.label==0).sum()}  "
      f"Pneumonia: {(rsna_binary.label==1).sum()}")

# Balance to ~same ratio as Kermany test (62% pneumonia)
rsna_pneu   = rsna_binary[rsna_binary.label==1]
rsna_normal = rsna_binary[rsna_binary.label==0]
n_normal    = int(len(rsna_pneu) * (234/390))  # match Kermany test ratio
rsna_normal = rsna_normal.sample(n=min(n_normal, len(rsna_normal)), random_state=42)
rsna_final  = pd.concat([rsna_pneu, rsna_normal]).reset_index(drop=True)

print(f"RSNA final  — Normal: {(rsna_final.label==0).sum()}  "
      f"Pneumonia: {(rsna_final.label==1).sum()}")
print(f"Total RSNA test samples: {len(rsna_final)}")

## Cell 5 — RSNA Dataset class (DICOM → tensor)
DICOM files store pixel arrays as 16-bit integers with varying window/level settings.
We normalise to [0,255] and convert to 3-channel RGB to match DenseNet input expectations.

In [ ]:
class RSNADataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        dcm_path = os.path.join(self.img_dir, f"{row['patientId']}.dcm")

        # Read DICOM → normalise → RGB PIL
        dcm      = pydicom.dcmread(dcm_path)
        pixels   = dcm.pixel_array.astype(np.float32)
        pixels   = ((pixels - pixels.min()) /
                    (pixels.max() - pixels.min() + 1e-8) * 255).astype(np.uint8)
        img      = Image.fromarray(pixels).convert("RGB")

        if self.transform:
            img = self.transform(img)
        return img, int(row["label"])

rsna_test_ds     = RSNADataset(rsna_final, RSNA_TRAIN_DIR, val_tf)
rsna_test_loader = DataLoader(rsna_test_ds, batch_size=32, shuffle=False,
                              num_workers=2, pin_memory=True)

print(f"RSNA test dataset: {len(rsna_test_ds)} images")
# Quick sanity check — load one batch
imgs, labels = next(iter(rsna_test_loader))
print(f"Batch shape: {imgs.shape}  Labels: {labels[:8].tolist()}")

## Cell 6 — Class imbalance & pos_weight

In [ ]:
from collections import Counter
import torch

device     = "cuda" if torch.cuda.is_available() else "cpu"
counts     = Counter([train_full.targets[i] for i in tr_idx])
pos_weight = torch.tensor([counts[1] / counts[0]]).to(device)

print(f"Normal: {counts[0]}  Pneumonia: {counts[1]}")
print(f"pos_weight = {pos_weight.item():.3f}")

## Cell 7 — MC Dropout DenseNet-121 with backbone dropout
Dropout at every DenseBlock (5 stochastic points total).
This produces meaningful uncertainty variance — unlike v4's head-only dropout.

In [ ]:
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class MCDropoutDenseNet(nn.Module):
    def __init__(self, dropout_p=0.3):
        super().__init__()
        base = models.densenet121(weights="IMAGENET1K_V1")

        self.conv0  = base.features.conv0
        self.norm0  = base.features.norm0
        self.pool0  = base.features.pool0

        self.block1 = base.features.denseblock1
        self.trans1 = base.features.transition1
        self.drop1  = nn.Dropout2d(p=dropout_p)

        self.block2 = base.features.denseblock2
        self.trans2 = base.features.transition2
        self.drop2  = nn.Dropout2d(p=dropout_p)

        self.block3 = base.features.denseblock3
        self.trans3 = base.features.transition3
        self.drop3  = nn.Dropout2d(p=dropout_p)

        self.block4 = base.features.denseblock4
        self.norm5  = base.features.norm5
        self.drop4  = nn.Dropout2d(p=dropout_p)

        self.drop_fc    = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(1024, 1)

    def forward(self, x):
        x = self.pool0(F.relu(self.norm0(self.conv0(x))))
        x = self.drop1(self.trans1(self.block1(x)))
        x = self.drop2(self.trans2(self.block2(x)))
        x = self.drop3(self.trans3(self.block3(x)))
        x = self.drop4(F.relu(self.norm5(self.block4(x))))
        x = F.adaptive_avg_pool2d(x, (1,1))
        x = self.drop_fc(torch.flatten(x, 1))
        return self.classifier(x)

def enable_mc_dropout(model):
    """Freeze BatchNorm (eval), keep all Dropout layers active (train)."""
    model.eval()
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()
    return model

model    = MCDropoutDenseNet(dropout_p=0.3).to(device)
n_drops  = sum(1 for m in model.modules()
               if isinstance(m, (nn.Dropout, nn.Dropout2d)))
print(f"{device} | params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Stochastic dropout points: {n_drops}")

## Cell 8 — Train on Kermany

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, verbose=True)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            if train: optimizer.zero_grad()
            out  = model(imgs).squeeze()
            loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item()
            preds   = (torch.sigmoid(out) > 0.5).long()
            correct += (preds == labels.long()).sum().item()
            total   += len(labels)
    return total_loss / len(loader), correct / total

best_val_loss, patience_ctr, patience = float("inf"), 0, 4

for epoch in range(25):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step(vl_loss)
    print(f"Ep {epoch+1:02d} | tr={tr_loss:.4f}/{tr_acc:.4f} | val={vl_loss:.4f}/{vl_acc:.4f}")
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss; patience_ctr = 0
        torch.save(model.state_dict(), "best_model_v5.pth")
        print(f"         ✓ Saved (val_loss={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stop at epoch {epoch+1}"); break

model.load_state_dict(torch.load("best_model_v5.pth"))
print("\nBest model loaded.")

## Cell 9 — MC Dropout inference on both test sets
Run T=50 passes on:
1. **Kermany test** — in-distribution (ID): model has seen this type of data
2. **RSNA test** — out-of-distribution (OOD): multi-site, real-world, unseen during training

**Key hypothesis:** Uncertainty should be significantly higher on RSNA (OOD) than Kermany (ID).
This is your primary research finding.

In [ ]:
def mc_predict(model, loader, T=50):
    enable_mc_dropout(model)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            passes = torch.stack([
                torch.sigmoid(model(imgs)) for _ in range(T)
            ])
            all_probs.append(passes.cpu().numpy())
            all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs,  axis=1).squeeze(-1)
    labels = np.concatenate(all_labels)
    return probs.mean(axis=0), probs.var(axis=0), labels

print("Running MC inference on Kermany test (ID)...")
mean_id, var_id, labels_id = mc_predict(model, kermany_test_loader, T=50)

print("Running MC inference on RSNA test (OOD)...")
mean_ood, var_ood, labels_ood = mc_predict(model, rsna_test_loader, T=50)

# ── Summary ───────────────────────────────────────────────────────────────
print()
print("="*55)
print(f"{'Metric':<30} {'ID (Kermany)':>10} {'OOD (RSNA)':>12}")
print("="*55)
print(f"{'Mean uncertainty':<30} {var_id.mean():>10.5f} {var_ood.mean():>12.5f}")
print(f"{'Std  uncertainty':<30} {var_id.std():>10.5f} {var_ood.std():>12.5f}")
print(f"{'Max  uncertainty':<30} {var_id.max():>10.5f} {var_ood.max():>12.5f}")
print("="*55)

ratio = var_ood.mean() / (var_id.mean() + 1e-10)
print(f"\nOOD/ID uncertainty ratio: {ratio:.2f}x")
print("Hypothesis confirmed :", var_ood.mean() > var_id.mean())

## Cell 10 — Uncertainty distribution: ID vs OOD
This is Figure 1 of your paper. The two distributions should be clearly separated —
RSNA (OOD) shifted right (higher uncertainty) vs Kermany (ID) clustered near zero.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: overlapping histograms ──────────────────────────────────────────
ax = axes[0]
ax.hist(var_id,  bins=50, alpha=0.6, color="#185FA5",
        label=f"Kermany ID  (μ={var_id.mean():.5f})",  density=True)
ax.hist(var_ood, bins=50, alpha=0.6, color="#E24B4A",
        label=f"RSNA OOD (μ={var_ood.mean():.5f})", density=True)
ax.set_xlabel("MC Dropout Uncertainty (predictive variance)", fontsize=11)
ax.set_ylabel("Density")
ax.set_title("Uncertainty Distribution\nIn-Distribution vs Out-of-Distribution", fontsize=11)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# ── Right: box plots ──────────────────────────────────────────────────────
ax = axes[1]
bp = ax.boxplot([var_id, var_ood],
                labels=["Kermany\n(ID)", "RSNA\n(OOD)"],
                patch_artist=True, notch=True,
                medianprops=dict(color="black", linewidth=2))
bp["boxes"][0].set_facecolor("#185FA5")
bp["boxes"][0].set_alpha(0.7)
bp["boxes"][1].set_facecolor("#E24B4A")
bp["boxes"][1].set_alpha(0.7)
ax.set_ylabel("Predictive Variance (uncertainty)")
ax.set_title("Uncertainty Spread\nID vs OOD", fontsize=11)
ax.grid(alpha=0.3, axis="y")

plt.suptitle("MC Dropout Uncertainty: In-Distribution vs Out-of-Distribution",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("uncertainty_id_vs_ood.png", dpi=150)
plt.show()

# Mann-Whitney U test — are the distributions significantly different?
from scipy import stats
stat, pval = stats.mannwhitneyu(var_id, var_ood, alternative="less")
print(f"Mann-Whitney U test (ID < OOD): p = {pval:.2e}")
print("Statistically significant (p<0.05):", pval < 0.05)

## Cell 11 — Optimal threshold via Youden's J
Fit on Kermany ID predictions (same distribution as training).

In [ ]:
from sklearn.metrics import roc_curve, classification_report, f1_score

fpr, tpr, thresholds = roc_curve(labels_id, mean_id)
youdens_j   = tpr + (1 - fpr) - 1
best_thresh = thresholds[np.argmax(youdens_j)]

print(f"Optimal threshold (Youden's J): {best_thresh:.3f}")
print()
print("── Kermany ID results ──────────────────────")
preds_id  = (mean_id  > best_thresh).astype(int)
print(classification_report(labels_id, preds_id,
    target_names=["Normal","Pneumonia"]))

print("── RSNA OOD results (same threshold) ───────")
preds_ood = (mean_ood > best_thresh).astype(int)
print(classification_report(labels_ood, preds_ood,
    target_names=["Normal","Pneumonia"]))

## Cell 12 — Three-way calibration study on both datasets
Calibration is fitted on Kermany val set, then evaluated on both ID and OOD test sets.
**Key finding:** Calibration methods trained on ID data may not transfer to OOD.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression

def ece(probs, labels, n_bins=10):
    bins = np.linspace(0,1,n_bins+1); err = 0.0
    for i in range(n_bins):
        mask = (probs>=bins[i])&(probs<bins[i+1])
        if mask.sum()==0: continue
        err += mask.sum()*abs(labels[mask].mean()-probs[mask].mean())
    return err/len(probs)

# ── Fit calibrators on val set ────────────────────────────────────────────
def get_logits(model, loader):
    model.eval(); logits, labs = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits.append(model(imgs.to(device)).squeeze().cpu())
            labs.append(labels)
    return torch.cat(logits), torch.cat(labs).float()

val_logits,  val_labs  = get_logits(model, val_loader)
test_id_logits, _      = get_logits(model, kermany_test_loader)
test_ood_logits, _     = get_logits(model, rsna_test_loader)

# Temperature scaling
class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.T = nn.Parameter(torch.ones(1))
    def forward(self, x): return x / self.T

scaler  = TempScaler()
ts_opt  = optim.LBFGS([scaler.T], lr=0.01, max_iter=50)
ts_crit = nn.BCEWithLogitsLoss()
def ts_step():
    ts_opt.zero_grad()
    loss = ts_crit(scaler(val_logits), val_labs)
    loss.backward(); return loss
ts_opt.step(ts_step)
T_learned = scaler.T.item()

temp_id  = torch.sigmoid(scaler(test_id_logits)).detach().numpy()
temp_ood = torch.sigmoid(scaler(test_ood_logits)).detach().numpy()

# Isotonic regression
val_probs = torch.sigmoid(val_logits).numpy()
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(val_probs, val_labs.numpy())
iso_id  = iso.predict(mean_id)
iso_ood = iso.predict(mean_ood)

# ── ECE table ────────────────────────────────────────────────────────────
print("="*60)
print(f"{'Method':<28} {'ID (Kermany)':>14} {'OOD (RSNA)':>14}")
print("="*60)
print(f"{'Uncalibrated':<28} {ece(mean_id,labels_id):>14.4f} {ece(mean_ood,labels_ood):>14.4f}")
print(f"{'Temperature (T={:.3f})':<28} {ece(temp_id,labels_id):>14.4f} {ece(temp_ood,labels_ood):>14.4f}".format(T_learned))
print(f"{'Isotonic regression':<28} {ece(iso_id,labels_id):>14.4f} {ece(iso_ood,labels_ood):>14.4f}")
print("="*60)

# ── Reliability diagrams: 2x3 grid ───────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
configs = [
    (mean_id,  labels_id,  "Uncalibrated — ID",     "#E24B4A"),
    (temp_id,  labels_id,  f"Temp Scaling (T={T_learned:.2f}) — ID", "#E88A2A"),
    (iso_id,   labels_id,  "Isotonic — ID",          "#639922"),
    (mean_ood, labels_ood, "Uncalibrated — OOD",    "#E24B4A"),
    (temp_ood, labels_ood, f"Temp Scaling (T={T_learned:.2f}) — OOD","#E88A2A"),
    (iso_ood,  labels_ood, "Isotonic — OOD",         "#639922"),
]
for ax,(probs,labs,title,color) in zip(axes.flat, configs):
    fp,mp = calibration_curve(labs, probs, n_bins=10)
    ax.plot([0,1],[0,1],"k--",alpha=0.5,label="Perfect")
    ax.plot(mp,fp,"o-",color=color,lw=2,label=f"ECE={ece(probs,labs):.3f}")
    ax.fill_between(mp,mp,fp,alpha=0.1,color=color)
    ax.set_title(title,fontweight="bold",fontsize=11)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
    ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1)

axes[0,0].set_ylabel("In-Distribution (Kermany)\nFraction positives", fontsize=10)
axes[1,0].set_ylabel("Out-of-Distribution (RSNA)\nFraction positives", fontsize=10)
plt.suptitle("Calibration: ID vs OOD — Three Methods",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("calibration_id_vs_ood.png", dpi=150)
plt.show()

## Cell 13 — Uncertainty vs accuracy: ID and OOD side by side

In [ ]:
def unc_vs_acc_plot(ax, var_p, preds, labels, title, color):
    bin_edges = np.percentile(var_p, np.linspace(0,100,6))
    bin_accs, bin_lbls = [], []
    for i in range(len(bin_edges)-1):
        lo,hi = bin_edges[i],bin_edges[i+1]
        mask  = (var_p>=lo)&(var_p<=hi) if i==len(bin_edges)-2                 else (var_p>=lo)&(var_p<hi)
        if mask.sum()==0: continue
        bin_accs.append((preds[mask]==labels[mask]).mean())
        bin_lbls.append(f"Q{i+1}\n(n={mask.sum()})")
    bars = ax.bar(bin_lbls, bin_accs, color=color, alpha=0.8, edgecolor="white")
    ax.axhline(0.5, color="red", linestyle="--", lw=1, label="Chance")
    ax.axhline(np.mean(bin_accs), color="grey", linestyle=":", lw=1,
               label=f"Mean={np.mean(bin_accs):.2f}")
    ax.set_ylim(0,1.1); ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Uncertainty quintile (Q1=confident, Q5=uncertain)")
    ax.set_ylabel("Accuracy"); ax.legend(fontsize=9)
    for bar,acc in zip(bars,bin_accs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{acc:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    return bin_accs

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
acc_id  = unc_vs_acc_plot(axes[0], var_id,  preds_id,  labels_id,
                           "ID (Kermany) — Acc vs Uncertainty", "#185FA5")
acc_ood = unc_vs_acc_plot(axes[1], var_ood, preds_ood, labels_ood,
                           "OOD (RSNA) — Acc vs Uncertainty", "#E24B4A")

plt.suptitle("Accuracy vs Uncertainty: In-Distribution vs Out-of-Distribution",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("acc_vs_unc_id_ood.png", dpi=150)
plt.show()

# Check signal direction
print("ID  — unc(correct) vs unc(incorrect):",
      f"{var_id[preds_id==labels_id].mean():.5f} vs "
      f"{var_id[preds_id!=labels_id].mean():.5f}  "
      f"| signal {'✓ OK' if var_id[preds_id!=labels_id].mean() > var_id[preds_id==labels_id].mean() else '✗ inverted'}")
print("OOD — unc(correct) vs unc(incorrect):",
      f"{var_ood[preds_ood==labels_ood].mean():.5f} vs "
      f"{var_ood[preds_ood!=labels_ood].mean():.5f}  "
      f"| signal {'✓ OK' if var_ood[preds_ood!=labels_ood].mean() > var_ood[preds_ood==labels_ood].mean() else '✗ inverted'}")

## Cell 14 — Clinical Triage Simulation on OOD (RSNA) ⭐
This is the primary contribution. On OOD data, uncertainty correctly identifies
hard cases — deferring them to a radiologist meaningfully improves performance.
Compare the improvement on OOD vs the flat curve on ID (v4 result).

In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score

def triage_sim(var_p, mean_p, preds, labels, label):
    sorted_idx  = np.argsort(var_p)
    defer_rates = np.arange(0, 0.55, 0.05)
    results = []
    for dr in defer_rates:
        n_keep = int(len(var_p)*(1-dr))
        keep   = sorted_idx[:n_keep]
        if len(keep)==0: continue
        y_t,y_p,y_prob = labels[keep],preds[keep],mean_p[keep]
        acc  = (y_p==y_t).mean()
        rec  = recall_score(y_t,y_p,zero_division=0)
        spec = (y_p[y_t==0]==0).mean() if (y_t==0).sum()>0 else 0
        auc  = roc_auc_score(y_t,y_prob) if len(np.unique(y_t))>1 else 0
        results.append(dict(deferral=dr,n_keep=n_keep,
            accuracy=acc,recall=rec,specificity=spec,auc=auc))
    return results

res_id  = triage_sim(var_id,  mean_id,  preds_id,  labels_id,  "ID")
res_ood = triage_sim(var_ood, mean_ood, preds_ood, labels_ood, "OOD")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, title, color in [
    (axes[0], res_id,  "ID (Kermany) — Triage",  "#185FA5"),
    (axes[1], res_ood, "OOD (RSNA)  — Triage",   "#E24B4A"),
]:
    dr_v = [r["deferral"]*100 for r in res]
    ac_v = [r["accuracy"]*100 for r in res]
    sp_v = [r["specificity"]*100 for r in res]
    re_v = [r["recall"]*100 for r in res]

    ax.plot(dr_v, ac_v, "o-", color=color, lw=2.5, ms=7, label="Accuracy")
    ax.plot(dr_v, sp_v, "s--", color="#639922", lw=2, ms=6, label="Specificity")
    ax.plot(dr_v, re_v, "^:", color="#E88A2A", lw=2, ms=6, label="Pneu recall")
    ax.axhline(ac_v[0], color="grey", linestyle=":", lw=1,
               label=f"Baseline={ac_v[0]:.1f}%")
    ax.fill_between(dr_v, ac_v[0], ac_v, alpha=0.1, color=color)
    ax.set_xlabel("Cases deferred to radiologist (%)")
    ax.set_ylabel("Performance (%)")
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.legend(fontsize=9); ax.set_ylim(40,105); ax.grid(alpha=0.3)

plt.suptitle("Clinical Triage Simulation — ID vs OOD",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("triage_id_vs_ood.png", dpi=150)
plt.show()

## Cell 15 — Triage summary tables

In [ ]:
def print_triage_table(results, title):
    print(f"\n{title}")
    print("="*75)
    print(f"{'Deferral':>9} {'Decided':>8} {'Accuracy':>10} "
          f"{'Recall':>8} {'Specificity':>13} {'AUC':>8}")
    print("="*75)
    for r in results:
        flag = " ◄" if r["deferral"] in [0.0,0.10,0.20,0.30] else ""
        print(f"  {r['deferral']*100:>6.0f}%  {r['n_keep']:>8}  "
              f"{r['accuracy']*100:>8.1f}%  {r['recall']*100:>6.1f}%  "
              f"{r['specificity']*100:>11.1f}%  {r['auc']:>6.3f}{flag}")
    print("="*75)
    r0  = results[0]
    r20 = next(r for r in results if abs(r["deferral"]-0.20)<0.01)
    gain = (r20["accuracy"]-r0["accuracy"])*100
    print(f"  Accuracy gain at 20% deferral: {gain:+.1f}pp")

print_triage_table(res_id,  "ID (Kermany) — Triage")
print_triage_table(res_ood, "OOD (RSNA)  — Triage")

print("\n" + "="*55)
print("KEY FINDINGS FOR PAPER:")
r0_id  = res_id[0];  r20_id  = next(r for r in res_id  if abs(r["deferral"]-0.20)<0.01)
r0_ood = res_ood[0]; r20_ood = next(r for r in res_ood if abs(r["deferral"]-0.20)<0.01)
print(f"• ID  baseline AUC : {r0_id['auc']:.3f}")
print(f"• OOD baseline AUC : {r0_ood['auc']:.3f}")
print(f"• ID  accuracy gain at 20% deferral : {(r20_id['accuracy'] -r0_id['accuracy'])*100:+.1f}pp")
print(f"• OOD accuracy gain at 20% deferral : {(r20_ood['accuracy']-r0_ood['accuracy'])*100:+.1f}pp")
print(f"• Mean uncertainty ID  : {var_id.mean():.5f}")
print(f"• Mean uncertainty OOD : {var_ood.mean():.5f}")
print(f"• OOD/ID uncertainty ratio: {var_ood.mean()/(var_id.mean()+1e-10):.2f}x")
print("="*55)

## Cell 16 — Final metrics summary

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report

print("="*60)
print("FINAL RESULTS — v5 Kermany→RSNA Cross-Dataset")
print("="*60)

for tag,mean_p,labels,preds in [
    ("ID  (Kermany)", mean_id,  labels_id,  preds_id),
    ("OOD (RSNA)   ", mean_ood, labels_ood, preds_ood),
]:
    auc = roc_auc_score(labels, mean_p)
    f1  = f1_score(labels, preds)
    e   = ece(mean_p, labels)
    print(f"\n── {tag} ────────────────────────────")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  ECE (raw): {e:.4f}")
    print(f"  Mean unc : {(var_id if 'Kermany' in tag else var_ood).mean():.5f}")
    print(classification_report(labels, preds,
        target_names=["Normal","Pneumonia"]))

## Cell 17 — Grad-CAM: ID vs OOD comparison
Show 3 low-uncertainty ID cases (focused heatmaps) vs
3 high-uncertainty OOD cases (diffuse heatmaps).
This is Figure 4 of your paper — visual evidence of distribution shift.

In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model=model; self.gradients=None; self.activations=None
        target_layer.register_forward_hook(
            lambda m,i,o: setattr(self,"activations",o))
        target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,"gradients",go[0]))

    def generate(self, img_t):
        enable_mc_dropout(self.model)
        out=self.model(img_t); self.model.zero_grad()
        out[0,0].backward()
        w  =self.gradients.mean(dim=[2,3],keepdim=True)
        cam=(w*self.activations).sum(dim=1).squeeze()
        cam=torch.relu(cam).cpu().detach().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8)

def overlay_cam(img_t, img_raw, label, pred, unc, thresh, ax_orig, ax_cam):
    cam  = gcam.generate(img_t.unsqueeze(0).to(device))
    cam_r= cv2.resize(cam,(224,224))
    heat = cv2.applyColorMap((cam_r*255).astype(np.uint8),cv2.COLORMAP_JET)
    orig = (img_raw.permute(1,2,0).numpy()*255).astype(np.uint8)
    overlay=cv2.addWeighted(orig,0.6,heat,0.4,0)
    correct=int(pred>thresh)==label
    ax_orig.imshow(orig,cmap="gray"); ax_orig.axis("off")
    ax_orig.set_title(f"{'Pneumonia' if label else 'Normal'}",fontsize=9)
    ax_cam.imshow(overlay); ax_cam.axis("off")
    ax_cam.set_title(f"{'✓' if correct else '✗'} p={pred:.2f} u={unc:.4f}",fontsize=9)

gcam = GradCAM(model, model.block4)

# Raw (un-normalised) datasets for display
raw_display_tf = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor()])
kermany_raw = ImageFolder(os.path.join(KERMANY_BASE,"test"), transform=raw_display_tf)
rsna_raw    = RSNADataset(rsna_final, RSNA_TRAIN_DIR, raw_display_tf)

# Pick 3 low-unc ID + 3 high-unc OOD
low_id_idx  = np.argsort(var_id)[:3]
high_ood_idx= np.argsort(var_ood)[-3:]

fig, axes = plt.subplots(4, 6, figsize=(18, 12))
fig.suptitle("Grad-CAM: Low-Uncertainty ID vs High-Uncertainty OOD",
             fontsize=13, fontweight="bold")

for col, i in enumerate(low_id_idx):
    img_t,  lbl = kermany_test_ds[i]
    img_raw, _  = kermany_raw[i]
    axes[0,col*2].set_ylabel("ID orig", fontsize=9)
    overlay_cam(img_t,img_raw,lbl,mean_id[i],var_id[i],best_thresh,
                axes[0,col*2], axes[1,col*2])
    axes[0,col*2+1].axis("off"); axes[1,col*2+1].axis("off")

for col, i in enumerate(high_ood_idx):
    img_t,  lbl = rsna_test_ds[i]
    img_raw, _  = rsna_raw[i]
    axes[2,col*2].set_ylabel("OOD orig", fontsize=9)
    overlay_cam(img_t,img_raw,lbl,mean_ood[i],var_ood[i],best_thresh,
                axes[2,col*2], axes[3,col*2])
    axes[2,col*2+1].axis("off"); axes[3,col*2+1].axis("off")

plt.tight_layout()
plt.savefig("gradcam_id_vs_ood.png", dpi=150)
plt.show()